# درس 8: تمیز کردن داده و resample

متن کامل درس: [https://quantacademy.ir/courses/python-for-traders/cleaning-resample/](https://quantacademy.ir/courses/python-for-traders/cleaning-resample/)

داده‌های این دوره **شبیه‌سازی‌شده** هستند، نه قیمت واقعی بازار؛ هدف یاد دادن روش است. آموزشی است و توصیه سرمایه‌گذاری نیست.

سلول‌ها را از بالا به پایین اجرا کن (Shift+Enter).

## ۱. داده واقعی کثیف است

## ۲. خواندن و اولین نگاه

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/quantacademyir/python-for-traders/main/data/btc_hourly_sample.csv"
raw = pd.read_csv(URL)
print(raw.head(3))
print(raw.shape)

In [ ]:
raw["timestamp"] = pd.to_datetime(raw["timestamp"])
df = raw.set_index("timestamp")

## ۳. بازرسی: چه ایرادی دارد؟

In [ ]:
print(df.index.is_monotonic_increasing)

In [ ]:
df = df.sort_index()
print(df.index.is_monotonic_increasing)

In [ ]:
print(df.isna().sum().to_dict())

In [ ]:
print(df.index.duplicated().sum())
print(df[df.index.duplicated(keep=False)])

In [ ]:
df = df[~df.index.duplicated(keep="first")]
print(df.shape)

In [ ]:
full = df.asfreq("h")
print(full.shape)
print(full["close"].isna().sum())
missing = full[full["close"].isna()].index
print(list(missing.strftime("%m-%d %H:%M")))

In [ ]:
print((df["close"] <= 0).sum())
print(df[df["close"] <= 0])

In [ ]:
df.loc[df["close"] <= 0, "close"] = float("nan")
full = df.asfreq("h")
print(full["close"].isna().sum())

## ۴. با جاهای خالی چه کنیم؟

In [ ]:
clean = full.dropna(subset=["close"])
print(clean.shape)

In [ ]:
filled = full.copy()
filled["close"] = filled["close"].ffill()
print(filled["close"].isna().sum())

## ۵. resample: ساعتی به روزانه

In [ ]:
daily = full.resample("D").agg({
    "open": "first",
    "high": "max",
    "low": "min",
    "close": "last",
    "volume": "sum",
})
print(daily.head(3).round(1))
print(daily.shape)

In [ ]:
counts = full["close"].resample("D").count()
print(counts.head(5).tolist(), counts.min())

## ۶. چند نکته درباره کدهای فرکانس

## ۷. ساعت جهانی (UTC) و روز معاملاتی

In [ ]:
tehran = full.tz_convert("Asia/Tehran")
print(full.index.tz)
print(tehran.index[0])
print(tehran.resample("D").size().head(3).tolist())